In [1]:
import pandas as pd
import numpy as np
import os
import json
import datetime
import gc
from tqdm import tqdm
import warnings
import re  # ✅ 텍스트 정규화를 위한 정규식 사용

warnings.filterwarnings("ignore")

from pycaret.regression import *
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


class MercariPyCaretAnalyzer:
    """
    Mercari Price Suggestion Challenge용 PyCaret 분석기

    주요 기능:
    1) 데이터 로딩 및 전처리
       - price 필터링 및 로그 변환(log1p)
       - category_name을 대/중/소로 분해
       - 희귀 브랜드/카테고리를 'Other_*'로 통합 (rare category collapsing)
       - 결측 텍스트 채우기 및 텍스트 길이 기반 수치 피처 생성

    2) 텍스트 벡터화 + 차원 축소
       - simple_normalize()로 텍스트 정규화 후 *_clean 컬럼 생성
       - TF-IDF + n-gram((1,2)) + TruncatedSVD로 dense 벡터 생성
       - 카테고리/브랜드/배송 + 텍스트 길이 피처를 그대로 붙여서 최종 피처 구성

    3) PyCaret setup & compare_models
       - PyCaret 3.2.2 사용
       - 이미 price를 로그 변환했으므로 transformation=False
       - categorical_features에 문자열/카테고리 컬럼을 그대로 넘겨서
         PyCaret 내부 인코더를 활용

    4) 모델 성능 저장
       - predict_model() 결과의 Label(로그 스케일)을 expm1으로 되돌려
         원래 가격 스케일에서 R2 / RMSE / MAE 계산 후 JSON 저장

    5) Test 예측 & submission 생성
       - test_vectorized에 대해 predict_model
       - 로그 예측값을 expm1으로 되돌려 price로 저장
    """

    def __init__(
        self,
        data_dir="../data",
        images_dir="../images",
        results_dir="../results",
    ):
        self.data_dir = data_dir
        self.images_dir = images_dir
        self.results_dir = results_dir

        self.train = None
        self.test = None
        self.best_model = None
        self.setup_result = None
        self.metrics = {}

        os.makedirs(self.images_dir, exist_ok=True)
        os.makedirs(self.results_dir, exist_ok=True)

    # ------------------------------------------------------------------
    # 🔹 (공통 유틸) 희귀 카테고리/브랜드를 "Other" 그룹으로 묶는 함수
    # ------------------------------------------------------------------
    def _collapse_rare_values(self, col, top_k, rare_label="Other"):
        """
        col: 희귀 값 통합 대상 컬럼명 (예: 'brand_name', 'main_cat', ...)
        top_k: 등장 빈도 기준 상위 몇 개만 유지할지
        rare_label: 나머지(희귀값)를 치환할 문자열
        """
        combined = pd.concat([self.train[col], self.test[col]], axis=0)
        value_counts = combined.value_counts()

        # 상위 top_k 값만 유지하고, 나머지는 rare_label로 통합
        top_values = value_counts.index[:top_k]

        self.train[col] = self.train[col].where(
            self.train[col].isin(top_values), rare_label
        )
        self.test[col] = self.test[col].where(
            self.test[col].isin(top_values), rare_label
        )

    # ------------------------------------------------------------------
    # 🔹 (공통 유틸) 텍스트 간단 정규화 함수
    # ------------------------------------------------------------------
    def _simple_normalize(self, text: str) -> str:
        """
        Mercari 1등 코드 철학을 단순화한 텍스트 정규화 함수
        - 모두 소문자로 변환
        - 특정 특수문자(_ - . /)를 공백으로 치환
        - 숫자는 전부 'num' 토큰으로 치환
        - 다중 공백 정리
        """
        text = str(text).lower()
        # 특수문자 → 공백
        text = re.sub(r"[_\-\./]", " ", text)
        # 숫자를 'num'으로
        text = re.sub(r"\d+", " num ", text)
        # 공백 정리
        text = re.sub(r"\s+", " ", text).strip()
        return text

    # ------------------------------------------------------------------
    # 1. 데이터 로딩 + 기본 전처리 + 희귀 카테고리/브랜드 통합 + 길이 피처 생성
    # ------------------------------------------------------------------
    def load_data(self, train_file="train.tsv", test_file="test.tsv", sep="\t"):
        print("📂 데이터 로딩 시작...")
        train_path = os.path.join(self.data_dir, train_file)
        test_path = os.path.join(self.data_dir, test_file)

        self.train = pd.read_csv(train_path, sep=sep)
        self.test = pd.read_csv(test_path, sep=sep)

        print(f"original data shape : train {self.train.shape}, test {self.test.shape}")
        print("✅ Price 외 결측치 처리 및 데이터 전처리 시작...")

        # --------------------------------------------------------------
        # 1-1) price 0 제거 + NaN 제거 (train만 해당)
        # --------------------------------------------------------------
        self.train = self.train[self.train["price"] > 0].dropna(subset=["price"])
        print("Price NaN count:", self.train["price"].isna().sum())

        # --------------------------------------------------------------
        # 1-2) category_name → main_cat / sub_cat / sub_sub_cat 분해,
        #      텍스트 결측치 채우기, name 결측도 방지
        # --------------------------------------------------------------
        for df_name, df in [("train", self.train), ("test", self.test)]:
            # category_name이 "대/중/소" 형태이면 split, 아니면 'missing'
            df["main_cat"], df["sub_cat"], df["sub_sub_cat"] = zip(
                *df["category_name"].apply(
                    lambda x: (
                        x.split("/")
                        if isinstance(x, str) and "/" in x
                        else ["missing", "missing", "missing"]
                    )
                )
            )

            # 주요 텍스트 컬럼 결측치 채우기 + 문자열로 강제
            df["brand_name"] = df["brand_name"].fillna("Unknown").astype(str)
            df["category_name"] = df["category_name"].fillna("Unknown").astype(str)
            df["item_description"] = (
                df["item_description"].fillna("No description").astype(str)
            )
            df["name"] = df["name"].fillna("No name").astype(str)

            # train / test에 다시 반영 + 인덱스 리셋
            if df_name == "train":
                self.train = df.reset_index(drop=True)
            else:
                self.test = df.reset_index(drop=True)

        # --------------------------------------------------------------
        # 1-3) price 로그 변환 (log1p) - PyCaret 내부 transformation은 끔
        # --------------------------------------------------------------
        self.train["price"] = np.log1p(self.train["price"])

        # --------------------------------------------------------------
        # 1-4) 희귀 브랜드 / 카테고리 통합 (rare category collapsing)
        #      - 상위 N개만 유지, 나머지는 'Other_*'로 묶기
        # --------------------------------------------------------------
        print("📊 희귀 카테고리/브랜드 통합(rare category collapsing) 시작...")
        self._collapse_rare_values("brand_name", top_k=4500, rare_label="Other_brand")
        self._collapse_rare_values("main_cat", top_k=1000, rare_label="Other_main")
        self._collapse_rare_values("sub_cat", top_k=1000, rare_label="Other_sub")
        self._collapse_rare_values(
            "sub_sub_cat", top_k=1000, rare_label="Other_sub_sub"
        )

        # --------------------------------------------------------------
        # 1-5) 텍스트 길이 기반 수치 피처 추가
        #      - name_len_char / name_len_word
        #      - desc_len_char / desc_len_word
        # --------------------------------------------------------------
        for df_name, df in [("train", self.train), ("test", self.test)]:
            df["name_len_char"] = df["name"].astype(str).str.len()
            df["name_len_word"] = df["name"].astype(str).str.split().str.len()
            df["desc_len_char"] = df["item_description"].astype(str).str.len()
            df["desc_len_word"] = (
                df["item_description"].astype(str).str.split().str.len()
            )

        # --------------------------------------------------------------
        # 1-6) shipping / item_condition_id를 category 타입으로 캐스팅
        #      (PyCaret에 categorical_features로 넘길 예정)
        # --------------------------------------------------------------
        for df in [self.train, self.test]:
            df["shipping"] = df["shipping"].astype("category")
            df["item_condition_id"] = df["item_condition_id"].astype("category")

        print("Final Price NaN count:", self.train["price"].isna().sum())
        print("Train length:", len(self.train))
        print("\nTrain head:")
        print(self.train.head())

        print(f"\nTrain info:\n{'='*50}")
        print(self.train.info())

        print(
            f"\n✅ 데이터 로드 완료: train {self.train.shape}, test {self.test.shape}"
        )

    # ------------------------------------------------------------------
    # 2. 텍스트 벡터화 + 차원 축소 + 카테고리/길이 피처 결합
    # ------------------------------------------------------------------
    def vectorize_text(
        self,
        text_columns=["name", "item_description"],
        method="tfidf",
        max_features=50000,
        n_components=100,
    ):
        """
        text_columns: 벡터화할 텍스트 컬럼 리스트
        method: 'tfidf' 또는 'count'
        max_features: Vectorizer의 최대 피처 수
        n_components: TruncatedSVD 차원 수
        """
        print("📝 텍스트 벡터화 및 차원 축소 시작...")

        # --------------------------------------------------------------
        # 2-1) 텍스트 정규화 컬럼(*_clean) 생성
        #      - simple_normalize() 적용
        # --------------------------------------------------------------
        for col in text_columns:
            clean_col = f"{col}_clean"
            if clean_col not in self.train.columns:
                self.train[clean_col] = self.train[col].astype(str).apply(
                    self._simple_normalize
                )
                self.test[clean_col] = self.test[col].astype(str).apply(
                    self._simple_normalize
                )

        vectors = []
        feature_names = []

        # --------------------------------------------------------------
        # 2-2) 텍스트 컬럼별로 TF-IDF (또는 Count) + SVD 적용
        #      - ngram_range=(1,2)로 1~2그램 사용 (속도/성능 타협)
        # --------------------------------------------------------------
        for col in tqdm(text_columns, desc="Text columns"):
            print(f"▶ 컬럼: {col}")
            clean_col = f"{col}_clean"

            if method == "tfidf":
                vec = TfidfVectorizer(
                    max_features=max_features,
                    ngram_range=(1, 2),  # ✅ 1~2-gram 적용 (후보 4번)
                )
            elif method == "count":
                vec = CountVectorizer(
                    max_features=max_features,
                    ngram_range=(1, 2),
                )
            else:
                raise ValueError("method must be 'tfidf' or 'count'")

            # train + test를 합쳐서 fit 후, 다시 train/test로 나눠 transform
            combined_text = pd.concat(
                [self.train[clean_col], self.test[clean_col]], axis=0
            )
            vec.fit(combined_text)

            train_vec = vec.transform(self.train[clean_col])
            test_vec = vec.transform(self.test[clean_col])

            # ----------------------------------------------------------
            # 2-3) 차원 축소: TruncatedSVD
            #      - TF-IDF의 차원이 max_features보다 작으면 SVD 생략
            # ----------------------------------------------------------
            if n_components < train_vec.shape[1]:
                svd = TruncatedSVD(n_components=n_components, random_state=23)
                train_vec = svd.fit_transform(train_vec)
                test_vec = svd.transform(test_vec)
                print(f"   ▪ 차원 축소 완료: {train_vec.shape[1]} components")
            else:
                train_vec = train_vec.toarray()
                test_vec = test_vec.toarray()

            vectors.append((train_vec, test_vec))
            feature_names.append([f"{col}_{i}" for i in range(train_vec.shape[1])])

            # 메모리 해제
            del combined_text, vec
            gc.collect()

        # --------------------------------------------------------------
        # 2-4) 모든 텍스트 피처를 가로 방향으로 합치기
        # --------------------------------------------------------------
        train_features = np.hstack([v[0] for v in vectors])
        test_features = np.hstack([v[1] for v in vectors])

        self.train_vectorized = pd.DataFrame(
            train_features, columns=[f for sub in feature_names for f in sub]
        )
        self.test_vectorized = pd.DataFrame(
            test_features, columns=[f for sub in feature_names for f in sub]
        )

        # --------------------------------------------------------------
        # 2-5) 카테고리/브랜드/배송 + 텍스트 길이 피처를 그대로 붙이기
        #      (LabelEncoder 제거 -> PyCaret 내부 인코딩 사용, 후보 5번)
        # --------------------------------------------------------------
        categorical_cols = [
            "main_cat",
            "sub_cat",
            "sub_sub_cat",
            "brand_name",
            "item_condition_id",
            "shipping",
        ]

        numeric_length_cols = [
            "name_len_char",
            "name_len_word",
            "desc_len_char",
            "desc_len_word",
        ]

        for col in categorical_cols + numeric_length_cols:
            if col in self.train.columns:
                # 인덱스가 0~N-1로 맞춰져 있으므로 values 그대로 붙여도 안전
                self.train_vectorized[col] = (
                    self.train[col].reset_index(drop=True)
                )
                self.test_vectorized[col] = (
                    self.test[col].reset_index(drop=True)
                )

        print(
            f"✅ 벡터화 + 차원 축소 + 카테고리/길이 피처 추가 완료: "
            f"train {self.train_vectorized.shape}, test {self.test_vectorized.shape}"
        )

    # ------------------------------------------------------------------
    # 3. PyCaret setup
    # ------------------------------------------------------------------
    def setup_pycaret(self, session_id=23):
        if not hasattr(self, "train_vectorized"):
            raise ValueError("먼저 vectorize_text()를 실행하세요.")

        print("🔧 PyCaret setup 시작...")

        # PyCaret에 넘길 카테고리 피처 이름들
        categorical_cols = [
            "main_cat",
            "sub_cat",
            "sub_sub_cat",
            "brand_name",
            "item_condition_id",
            "shipping",
        ]
        existing_categorical = [
            col for col in categorical_cols if col in self.train_vectorized.columns
        ]

        # ✅ 이미 price를 log1p로 변환했기 때문에 transformation=False로 설정
        #    (중복 타깃 변환 방지)
        self.setup_result = setup(
            data=self.train_vectorized.assign(
                price=self.train["price"].reset_index(drop=True)
            ),
            target="price",
            session_id=session_id,
            categorical_features=existing_categorical if existing_categorical else None,
            normalize=True,
            transformation=False,
            verbose=True,
        )
        print("✅ PyCaret setup 완료")

    # ------------------------------------------------------------------
    # 4. Base model 탐색
    # ------------------------------------------------------------------
    def find_base_model(self, sort_metric="R2"):
        if self.setup_result is None:
            raise ValueError("먼저 setup_pycaret()를 실행하세요.")

        print("🔍 Base model 탐색 시작...")
        self.best_model = compare_models(sort=sort_metric, n_select=1)
        print(f"🏆 Best model 선택 완료: {self.best_model}")
        return self.best_model

    # ------------------------------------------------------------------
    # 5. 모델 성능 저장 (원래 가격 스케일에서 R2/RMSE/MAE 계산)
    # ------------------------------------------------------------------
    def save_metrics(self, metrics_dict=None, model_name=None):
        """
        ⚠️ 기존 코드 오류 수정:
        - 이전에는 predict_model() 결과에 'R2', 'RMSE', 'MAE' 컬럼이 있다고 가정했지만,
          PyCaret의 predict_model()은 개별 샘플에 대한 예측/잔차만 반환하고,
          전체 메트릭은 pull()에서 확인해야 함.
        - 따라서 여기서는 직접 y_true / y_pred를 꺼내와서
          sklearn.metrics로 R2 / RMSE / MAE를 계산한다.

        또한 price는 log1p(price_real)이므로,
        expm1()으로 되돌려 실제 가격 스케일에서 메트릭을 계산한다.
        """
        if metrics_dict is None:
            if self.best_model is None:
                raise ValueError("모델이 없습니다.")

            # train 전체에 대해 예측 수행
            pred_df = predict_model(self.best_model, data=self.train_vectorized.copy())

            # 로그 스케일 타깃/예측
            y_log_true = self.train["price"].values
            y_log_pred = pred_df["Label"].values

            # expm1으로 원래 가격 스케일로 되돌리기
            y_true = np.expm1(y_log_true)
            y_pred = np.expm1(y_log_pred)

            r2 = r2_score(y_true, y_pred)
            rmse = mean_squared_error(y_true, y_pred, squared=False)
            mae = mean_absolute_error(y_true, y_pred)

            metrics_dict = {
                "R2": round(r2, 4),
                "RMSE": round(rmse, 4),
                "MAE": round(mae, 4),
            }

        self.metrics = metrics_dict

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        if model_name is None:
            model_name = str(self.best_model).split("(")[0]
        file_path = os.path.join(
            self.results_dir, f"{model_name}_metrics_{timestamp}.json"
        )

        with open(file_path, "w") as f:
            json.dump(self.metrics, f, indent=4)

        print(f"💾 Metrics 저장 완료: {file_path}")

    # ------------------------------------------------------------------
    # 6. 시각화
    # ------------------------------------------------------------------
    def visualize_model(self, plots=["residuals", "feature"]):
        if self.best_model is None:
            raise ValueError("먼저 find_base_model()로 모델을 선택하세요.")

        print("🎨 시각화 시작...")
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        model_name = str(self.best_model).split("(")[0]

        for p in plots:
            try:
                # feature_importance 같은 alias를 쓰더라도 PyCaret 쪽 plot 이름에 맞게 사용할 수 있음
                plot_name = "feature" if p == "feature_importance" else p

                save_path = os.path.join(
                    self.images_dir, f"{model_name}_{plot_name}_{timestamp}.png"
                )
                plot_model(self.best_model, plot=plot_name, save=True)
                print(f"✅ {plot_name} plot 저장 완료: {save_path}")
            except Exception as e:
                print(f"⚠️ Plot {p} 실패: {e}")

    # ------------------------------------------------------------------
    # 7. Test 예측 & submission 생성
    # ------------------------------------------------------------------
    def predict_test(self, submission_file="submission.csv"):
        if self.best_model is None:
            raise ValueError("먼저 find_base_model()로 모델을 선택하세요.")

        print("📦 Test 데이터 예측 시작...")

        # test_vectorized에 대해 예측 (로그 스케일)
        predictions = predict_model(self.best_model, data=self.test_vectorized.copy())

        # ⚠️ 기존 코드 오류:
        # - 이전에는 predictions['Label'] (log 가격)을 그대로 submission에 price로 넣었음.
        # - Kaggle에서는 실제 가격 스케일이 필요하므로 expm1()으로 되돌려야 함.
        price_log_pred = predictions["Label"].values
        price_pred = np.expm1(price_log_pred)

        submission = pd.DataFrame(
            {"test_id": self.test["test_id"], "price": price_pred}
        )

        submission_path = os.path.join(self.results_dir, submission_file)
        submission.to_csv(submission_path, index=False)
        print(f"💾 Submission 저장 완료: {submission_path}")
        return submission

In [ ]:
# ----------------------------------------------------------------------
# 실행 예시 (노트북에서는 이 블록 대신 별도 셀에서 호출해도 됨)
# ----------------------------------------------------------------------
# analyzer = MercariPyCaretAnalyzer()
# analyzer.load_data(train_file="train.tsv", test_file="test.tsv")
# analyzer.vectorize_text(method="tfidf", max_features=50000, n_components=100)
# analyzer.setup_pycaret()
# analyzer.find_base_model(sort_metric="R2")
# analyzer.save_metrics()
# analyzer.visualize_model(plots=["residuals", "feature"])
# analyzer.predict_test(submission_file="submission.csv")


In [2]:
analyzer = MercariPyCaretAnalyzer()

In [3]:
analyzer.load_data(train_file="train.tsv", test_file="test.tsv")

📂 데이터 로딩 시작...
original data shape : train (1482535, 8), test (693359, 7)
✅ Price 외 결측치 처리 및 데이터 전처리 시작...
Price NaN count: 0
📊 희귀 카테고리/브랜드 통합(rare category collapsing) 시작...
Final Price NaN count: 0
Train length: 1481661

Train head:
   train_id                                 name item_condition_id  \
0         0  MLB Cincinnati Reds T Shirt Size XL                 3   
1         1     Razer BlackWidow Chroma Keyboard                 3   
2         2                       AVA-VIV Blouse                 1   
3         3                Leather Horse Statues                 1   
4         4                 24K GOLD plated rose                 1   

                                       category_name brand_name     price  \
0                                  Men/Tops/T-shirts    Unknown  2.397895   
1  Electronics/Computers & Tablets/Components & P...      Razer  3.970292   
2                        Women/Tops & Blouses/Blouse     Target  2.397895   
3                 Home/Home Décor/Ho

In [4]:
analyzer.vectorize_text(method="tfidf", max_features=50000, n_components=100)

📝 텍스트 벡터화 및 차원 축소 시작...


Text columns:   0%|          | 0/2 [00:00<?, ?it/s]

▶ 컬럼: name


Text columns:  50%|█████     | 1/2 [01:47<01:47, 107.52s/it]

   ▪ 차원 축소 완료: 100 components
▶ 컬럼: item_description


Text columns: 100%|██████████| 2/2 [07:32<00:00, 226.05s/it]

   ▪ 차원 축소 완료: 100 components


✅ 벡터화 + 차원 축소 + 카테고리/길이 피처 추가 완료: train (1481661, 210), test (693359, 210)


In [5]:
analyzer.setup_pycaret()

🔧 PyCaret setup 시작...


,Description,Value
0,Session id,23
1,Target,price
2,Target type,Regression
3,Original data shape,"(1481661, 211)"
4,Transformed data shape,"(1481661, 225)"
5,Transformed train set shape,"(1037162, 225)"
6,Transformed test set shape,"(444499, 225)"
7,Numeric features,204
8,Categorical features,6
9,Preprocess,True


✅ PyCaret setup 완료


In [ ]:
analyzer.find_base_model(sort_metric="R2")

In [ ]:
analyzer.save_metrics()

In [ ]:
analyzer.visualize_model(plots=["residuals", "feature"])

In [ ]:
analyzer.predict_test(submission_file="submission.csv")